# Qwen3-VL-8B -- Climate generalization pilot

Content-domain + chart-type generalization test (Chapter 7 Limitation 2, Chapter 8 item 4,
supervisor item 7). Solar-vs-wind line chart, fabricated data, climate-framed claim -- see
`climate_pilot/generate_climate_stimuli.py` for the stimulus design rationale. Same protocols
as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/` instead of `benchmarking/`. 25 posts (not 50/100) -- this is a scoped
pilot, not a full replication.

In [1]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.0.dev0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


✅ Done — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Tue Aug 18 15:26:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   29C    P0            123W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Loaded successfully")

Using device: cuda


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


✅ Loaded successfully


In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
from e1_utils.inference_qwen import run_inference_qwen

In [6]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_qwen import run_inference_with_scores_qwen

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [7]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_qwen)

✅ 001_correct → like
✅ 001_incorrect → scroll
✅ 002_correct → like
✅ 002_incorrect → scroll
✅ 003_correct → like
✅ 003_incorrect → scroll
✅ 004_correct → scroll
✅ 004_incorrect → scroll
✅ 005_correct → scroll
✅ 005_incorrect → scroll
✅ 006_correct → scroll
✅ 006_incorrect → scroll
✅ 007_correct → like
✅ 007_incorrect → scroll
✅ 008_correct → scroll
✅ 008_incorrect → scroll
✅ 009_correct → scroll
✅ 009_incorrect → scroll
✅ 010_correct → scroll
✅ 010_incorrect → scroll
✅ 011_correct → scroll
✅ 011_incorrect → scroll
✅ 012_correct → scroll
✅ 012_incorrect → scroll
✅ 013_correct → scroll
✅ 013_incorrect → scroll
✅ 014_correct → like
✅ 014_incorrect → scroll
✅ 015_correct → like
✅ 015_incorrect → scroll
✅ 016_correct → scroll
✅ 016_incorrect → scroll
✅ 017_correct → scroll
✅ 017_incorrect → scroll
✅ 018_correct → like
✅ 018_incorrect → scroll
✅ 019_correct → scroll
✅ 019_incorrect → scroll
✅ 020_correct → scroll
✅ 020_incorrect → scroll
✅ 021_correct → like
✅ 021_incorrect → scroll
✅ 022_co

In [8]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_qwen)

✅ 001_correct → like {'like': {'logprob': -16.349082946777344, 'prob_forced_choice': 0.34864513533394575}, 'scroll': {'logprob': -15.724082946777344, 'prob_forced_choice': 0.6513548646660542}}
✅ 001_incorrect → scroll {'like': {'logprob': -21.254079818725586, 'prob_forced_choice': 0.00407013385011074}, 'scroll': {'logprob': -15.75407886505127, 'prob_forced_choice': 0.9959298661498892}}
✅ 002_correct → like {'like': {'logprob': -16.599082946777344, 'prob_forced_choice': 0.37754066879814546}, 'scroll': {'logprob': -16.099082946777344, 'prob_forced_choice': 0.6224593312018546}}
✅ 002_incorrect → scroll {'like': {'logprob': -17.626930236816406, 'prob_forced_choice': 0.06754675120600638}, 'scroll': {'logprob': -15.001931190490723, 'prob_forced_choice': 0.9324532487939936}}
✅ 003_correct → like {'like': {'logprob': -16.849082946777344, 'prob_forced_choice': 0.32082130082460697}, 'scroll': {'logprob': -16.099082946777344, 'prob_forced_choice': 0.679178699175393}}
✅ 003_incorrect → scroll {'li

In [9]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,22.0
1,like_rate_correct_%,44.0
2,like_rate_incorrect_%,0.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,like
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
2,002_correct,correct,You are shown a social media post.\nYou can ei...,like
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
4,003_correct,correct,You are shown a social media post.\nYou can ei...,like
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
6,004_correct,correct,You are shown a social media post.\nYou can ei...,scroll
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
8,005_correct,correct,You are shown a social media post.\nYou can ei...,scroll
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/qwen3-vl-8b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [10]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_qwen)

✅ 001_correct_10 → like
✅ 001_incorrect_10 → scroll
✅ 002_correct_10 → like
✅ 002_incorrect_10 → like
✅ 003_correct_10 → like
✅ 003_incorrect_10 → scroll
✅ 004_correct_10 → scroll
✅ 004_incorrect_10 → scroll
✅ 005_correct_10 → like
✅ 005_incorrect_10 → scroll
✅ 006_correct_10 → scroll
✅ 006_incorrect_10 → scroll
✅ 007_correct_10 → like
✅ 007_incorrect_10 → scroll
✅ 008_correct_10 → scroll
✅ 008_incorrect_10 → scroll
✅ 009_correct_10 → like
✅ 009_incorrect_10 → scroll
✅ 010_correct_10 → scroll
✅ 010_incorrect_10 → scroll
✅ 011_correct_10 → scroll
✅ 011_incorrect_10 → scroll
✅ 012_correct_10 → scroll
✅ 012_incorrect_10 → scroll
✅ 013_correct_10 → scroll
✅ 013_incorrect_10 → scroll
✅ 014_correct_10 → like
✅ 014_incorrect_10 → scroll
✅ 015_correct_10 → like
✅ 015_incorrect_10 → scroll
✅ 016_correct_10 → scroll
✅ 016_incorrect_10 → scroll
✅ 017_correct_10 → like
✅ 017_incorrect_10 → scroll
✅ 018_correct_10 → like
✅ 018_incorrect_10 → scroll
✅ 019_correct_10 → like
✅ 019_incorrect_10 → scrol

In [11]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_qwen)

✅ 001_correct_10 → like {'like': {'logprob': -16.200946807861328, 'prob_forced_choice': 0.32082130082460697}, 'scroll': {'logprob': -15.450946807861328, 'prob_forced_choice': 0.679178699175393}}
✅ 001_incorrect_10 → scroll {'like': {'logprob': -16.951417922973633, 'prob_forced_choice': 0.10669059394565118}, 'scroll': {'logprob': -14.826417922973633, 'prob_forced_choice': 0.8933094060543487}}
✅ 002_correct_10 → like {'like': {'logprob': -16.126934051513672, 'prob_forced_choice': 0.5312093733737563}, 'scroll': {'logprob': -16.251934051513672, 'prob_forced_choice': 0.46879062662624377}}
✅ 002_incorrect_10 → like {'like': {'logprob': -16.200946807861328, 'prob_forced_choice': 0.32082130082460697}, 'scroll': {'logprob': -15.450946807861328, 'prob_forced_choice': 0.679178699175393}}
✅ 003_correct_10 → like {'like': {'logprob': -16.376934051513672, 'prob_forced_choice': 0.5}, 'scroll': {'logprob': -16.376934051513672, 'prob_forced_choice': 0.5}}
✅ 003_incorrect_10 → scroll {'like': {'logprob'

In [12]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,72.67
1,like_rate_correct_%,92.00
2,like_rate_incorrect_%,53.33


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,36.0,64.0,8.0
1,100,50.0,78.0,100.0,56.0
2,1000,50.0,94.0,96.0,92.0
3,10000,50.0,64.0,92.0,36.0
4,100000,50.0,86.0,100.0,72.0
5,1000000,50.0,78.0,100.0,56.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,like
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,like
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,like
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,scroll
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,like
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/qwen3-vl-8b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [13]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_qwen)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1015, please report a bug to PyTorch. 

In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")